# Pipeline Step 02: Network Topology & User Strata

This notebook imports our clean logic from `src/network.py` and runs the core user grouping and topology logic.
**Caching Guarantee**: It checks if the outputs already exist in `data/processed/` before calculating over the full corpus.

In [ ]:
import os
import pandas as pd
import sys

# Ensure we can import from src/
sys.path.append(os.path.abspath(os.path.join("..", "..")))
from src.network import (
    calculate_upvote_percentiles,
    build_insider_matrix,
    calculate_baseline_active_users,
    analyze_user_activity_percentiles,
    calculate_distribution_tiers
)

BASE_DIR = "/Users/nash/Projects/ConspiracyComments/"
RAW_COMMENTS_GLOB = os.path.join(BASE_DIR, "data/raw/r_conspiracy_comments*.jsonl*")
EMPATH_PARQUET = os.path.join(BASE_DIR, "data/processed/empath_scores_full.parquet")

TIERS_CACHE = os.path.join(BASE_DIR, "data/processed/tiers.csv")
INSIDER_MATRIX_CACHE = os.path.join(BASE_DIR, "data/processed/insider_matrix.csv")
BASELINE_USERS_CACHE = os.path.join(BASE_DIR, "data/processed/insider_vote_baselines.csv")

print("Ready.")

## 1. Network Activity Distribution (Tiers)

In [ ]:
if os.path.exists(TIERS_CACHE):
    print(f"✅ Cache found! Loading directly from {TIERS_CACHE}")
    df_tiers = pd.read_csv(TIERS_CACHE)
else:
    print(f"⚠️ Cache not found. Calculating distribution tiers...")
    df_tiers = calculate_distribution_tiers(RAW_COMMENTS_GLOB)
    df_tiers.to_csv(TIERS_CACHE, index=False)
    print(f"✅ Calculation complete and saved to cache!")
    
df_tiers

## 2. Insider Matrix

In [ ]:
if os.path.exists(INSIDER_MATRIX_CACHE):
    print(f"✅ Cache found! Loading directly from {INSIDER_MATRIX_CACHE}")
    df_insider_matrix = pd.read_csv(INSIDER_MATRIX_CACHE)
else:
    print(f"⚠️ Cache not found. Building insider matrix...")
    df_insider_matrix = build_insider_matrix(EMPATH_PARQUET)
    df_insider_matrix.to_csv(INSIDER_MATRIX_CACHE, index=False)
    print(f"✅ Matrix built and saved to cache!")
    
df_insider_matrix

## 3. Baseline Active Users (Timeline)

In [ ]:
if os.path.exists(BASELINE_USERS_CACHE):
    print(f"✅ Cache found! Loading directly from {BASELINE_USERS_CACHE}")
    df_baselines = pd.read_csv(BASELINE_USERS_CACHE)
else:
    print(f"⚠️ Cache not found. Calculating timeline baselines...")
    df_baselines = calculate_baseline_active_users(RAW_COMMENTS_GLOB)
    df_baselines.to_csv(BASELINE_USERS_CACHE, index=False)
    print(f"✅ Timelines generated and saved to cache!")
    
df_baselines.tail()